In [33]:
# REQUIRES
# ========
# System libraries
# ----------------
import time

# Third-party libraries
# ----------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# MODULES
# ========
# System modules
# --------------
from typing import Union
from pathlib import Path
from datetime import datetime

# Custom modules
# Defines
# -------
from src.config.globals import *

# Helpers functions
# -----------------
from src.helpers import funcs as hlp



In [34]:
# CHECK CHANGES ONF IMPORT FILES
# ==============================
import importlib
importlib.reload(hlp)

print(hlp.__file__)
print(hasattr(hlp, "SectionBegin"))
print([name for name in dir(hlp) if "Section" in name])

/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/funcs.py
True
['SectionBegin', 'SectionBreak', 'SectionEnd']


In [35]:
# PATHS
# =====
# CURRENT FILE PATH
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR.parent / "data").resolve()

# Global variables definition for this notebook
# ---------------------------------------------
# dataset:str = "../data/ventas_con_problemas.csv"
dataset: str = DATA_DIR / "ventas_con_problemas.csv"

# Pandas dataframe to hold the dataset
df: pd.DataFrame

In [36]:
# LOAD SOURCE DATASET (from CSV file)
# ===================================
hlp.SectionBegin(f"DATASET PATH: \n{dataset}", 0)
hlp.SectionBreak(0, 0)

try:
  df = pd.read_csv(dataset)
  print(f"File loaded successfully. Total of original rows: {len(df)}")
  print(f"First 5 rows of the dataset:")
  print(df.head())
  
except FileNotFoundError:
  print(f"Error: Please, check path. \n'{dataset}'")
  print(f"Error: File not found.")

hlp.SectionEnd(0)


+ =========================================================
+ DATASET PATH: 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/activities/RA1/lesson_02/data/ventas_con_problemas.csv
+ ---------------------------------------------------------
File loaded successfully. Total of original rows: 320
First 5 rows of the dataset:
  fecha             producto  cantidad  precio_unitario   total
0   NaN  Monitor 24 pulgadas       1.0           114.18  114.18
1   NaN      Cargador rapido       6.0            37.74  226.44
2   NaN  Monitor 24 pulgadas       5.0           193.91  969.55
3   NaN     Altavoz portatil       8.0           111.84  894.72
4   NaN      Cargador rapido       2.0            15.47   30.94
+ =========================================================


In [37]:
# DATA CLEANING
# =============
# 1. CHECK OF NULL VALUES
# -----------------------
tInit: float = time.time()

# CheckNullValues
# ------------------------
# Totalize and print the number of null values per column in the dataset, 
# as well as the total number of rows with at least one null value.
# 
# @param pPf: pd.DataFrame The pandas dataframe to check for null values.
# @return None
def CheckNullValues(pPf: pd.DataFrame) -> None:
  # Total null values per column in the dataset
  nullValuesPerColumn: int = pPf.isnull().sum()
  print("Null values per column:")
  print(nullValuesPerColumn)
  hlp.SectionBreak(0,0)

  # Total rows with at least one null value
  rowsWithNulls: pd.DataFrame = pPf[pPf.isnull().any(axis=1)]
  print(f"Total rows with at least one null value: \n{len(rowsWithNulls)}")

# Execute script
hlp.SectionBegin("CHECK OF NULL VALUES", 0)
hlp.SectionBreak(0,0)

CheckNullValues(df)
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)

+ =========================================================
+ CHECK OF NULL VALUES
+ ---------------------------------------------------------
Null values per column:
fecha              5
producto           5
cantidad           5
precio_unitario    5
total              5
dtype: int64
+ ---------------------------------------------------------
Total rows with at least one null value: 
25
+ ---------------------------------------------------------
+ Time to load: 0.0032 seconds
+ =========================================================


In [38]:
# DATA CLEANING
# =============
# 2. CHECK OF INVALID DATE VALUES
# -------------------------------
tInit: float = time.time()

# CheckInvalidDates
# -----------------
# Try to convert the date column to proper datetime format
# errors='coerce' converts all invalid dates to NaN
# 
# @param string pColumn Name of the column to convert to proper date format
# @return None
def CheckInvalidDates(pPf: pd.DataFrame, pColumn:str = 'fecha') -> None:
  properDateTime: pd.Series = pd.to_datetime(pPf[pColumn], errors='coerce')
  
  # How many dates were successfully converted and how many failed
  totRows: int = len(pPf)
  totProperDates: int = properDateTime.notna().sum()
  totInvalidDates: int = properDateTime.isna().sum()

  print(f"Total rows: {totRows}")
  print(f"Valid dates converted successfully: {totProperDates}")
  print(f"Invalid or null dates detected: {totInvalidDates}")

  # Show some of the rows with dates that failed
  print("\nExamples of rows with invalid dates:")
  print(df[properDateTime.isna()][pColumn].head(10))  
  
# Execute script
hlp.SectionBegin("CHECK OF INVALID DATE VALUES", 0)
print("+ Summary of conversion date column state:")
hlp.SectionBreak(0,0)

CheckInvalidDates(df, "fecha")
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)

+ =========================================================
+ CHECK OF INVALID DATE VALUES
+ Summary of conversion date column state:
+ ---------------------------------------------------------
Total rows: 320
Valid dates converted successfully: 302
Invalid or null dates detected: 18

Examples of rows with invalid dates:
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
25     2026-02-30
26     2026-13-10
27     2026-10-45
28    no_es_fecha
29     31/02/2026
Name: fecha, dtype: str
+ ---------------------------------------------------------
+ Time to load: 0.0044 seconds
+ =========================================================


In [47]:
# DATA CLEANING
# =============
# 3. CHECK OF DUPLICATED VALUES
# -----------------------------
tInit: float = time.time()

# CheckDuplicatedValues
# ---------------------
# Count exactly how many rows are duplicated, taking all columns in account
# 
# @param pd.DataFrame pPf The pandas dataframe to check
# @return None
def CheckDuplicatedValues(pPf: pd.DataFrame) -> None:
  cols:list[str] = ['fecha', 'producto', 'cantidad', 'precio_unitario', 'total']
  totDupl: int = pPf.duplicated(subset=cols).sum()  

  # Show a little snapshot
  if totDupl > 0:
    print(f"Snapshot of duplicated rows: ({totDupl})")
    hlp.nbsp()
    
    # keep=False show all instances of duplicates
    rowsDupl: pd.DataFrame = pPf[pPf.duplicated(keep=False)]
    print(rowsDupl.sort_values(by=cols)[cols].head(5))
  else:
    print("No duplicated rows found")
  
hlp.SectionBegin("CHECK OF DUPLICATED VALUES", 0)
print("+ Summary of fully duplicate rows found")
hlp.SectionBreak(0,0)
CheckDuplicatedValues(df)
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)

+ =========================================================
+ CHECK OF DUPLICATED VALUES
+ Summary of fully duplicate rows found
+ ---------------------------------------------------------
Snapshot of duplicated rows: (12)
 
          fecha          producto  cantidad  precio_unitario   total
117  2026-09-24      Funda tablet       2.0            11.14   22.28
315  2026-09-24      Funda tablet       2.0            11.14   22.28
116  2026-10-08  Teclado mecanico       2.0           103.16  206.32
282  2026-10-08  Teclado mecanico       2.0           103.16  206.32
301  2026-10-08  Teclado mecanico       2.0           103.16  206.32
+ ---------------------------------------------------------
+ Time to load: 0.0138 seconds
+ =========================================================


In [48]:
# DATA CLEANING
# =============
# 4. CHECK OF TOTAL COLUMN VALUE
# ------------------------------
tInit: float = time.time()

# CheckTotalColumnValues
# ----------------------
# Convert columns to numeric and check for negative or zero values
# 
# @param pd.DataFrame pPf The pandas dataframe to check
# @return None
def CheckTotalColumnValues(pPf: pd.DataFrame) -> None:
  # Ensure numeric conversion
  pPf['cantidad'] = pd.to_numeric(pPf['cantidad'], errors='coerce')
  pPf['precio_unitario'] = pd.to_numeric(pPf['precio_unitario'], errors='coerce')
  pPf['total'] = pd.to_numeric(pPf['total'], errors='coerce')

  # Get the expected Total
  pPf['total_temp'] = pPf['cantidad'] * pPf['precio_unitario']

  # Filter rows where both total amountnot do not match (using np.isclose with tolerance of 0.01)
  # Only check rows where any columns are null
  maskNoNull = pPf['cantidad'].notna() & pPf['precio_unitario'].notna() & pPf['total'].notna()
  inconsistencies = pPf[maskNoNull & (~np.isclose(pPf['total_temp'], pPf['total'], atol=0.01))]

  print(f"Rows with different totals: {len(inconsistencies)}")
  hlp.SectionBreak(0,0)
  print("Snapshot of rows with inconsistent total calculation:")
  print(inconsistencies[['producto', 'cantidad', 'precio_unitario', 'total', 'total_temp']].head(10))
  
hlp.SectionBegin("CHECK OF TOTAL COLUMN VALUE", 0)
print("+ Summary of rows with incorrect Total amount:")
hlp.SectionBreak(0,0)
CheckTotalColumnValues(df)
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)


+ =========================================================
+ CHECK OF TOTAL COLUMN VALUE
+ Summary of rows with incorrect Total amount:
+ ---------------------------------------------------------
Rows with different totals: 34
+ ---------------------------------------------------------
Snapshot of rows with inconsistent total calculation:
             producto  cantidad  precio_unitario   total  total_temp
59   Teclado mecanico       3.0            69.24 -207.72      207.72
60      Disco SSD 1TB       3.0           120.63 -361.89      361.89
61       Funda tablet       3.0            34.92 -104.76      104.76
62   Altavoz portatil       1.0            54.69  -54.69       54.69
63   Altavoz portatil       4.0            39.26 -157.04      157.04
64   Teclado mecanico       4.0            98.12 -392.48      392.48
65            Hub USB       3.0            16.89  -50.67       50.67
66  Raton inalambrico       2.0            41.59  -83.18       83.18
67   Teclado mecanico       3.0      

In [49]:
# DATA CLEANING
# =============
# 5. CHECK OF NEGATIVE OR ZERO NUMERIC VALUES
# -------------------------------------------
tInit: float = time.time()

# CheckNumericValues
# -------------------
# Convert columns to numeric and check for negative or zero values
# 
# @param pd.DataFrame pPf The pandas dataframe to check
# @return None
def CheckNumericValues(pPf: pd.DataFrame) -> None:
  # dfTmp: pd.DataFrame
  # Convert to numeric by forcing errors to NaN values for security
  pPf['cantidad_num'] = pd.to_numeric(pPf['cantidad'], errors='coerce')
  pPf['precio_unitario_num'] = pd.to_numeric(pPf['precio_unitario'], errors='coerce')
  pPf['total_num'] = pd.to_numeric(pPf['total'], errors='coerce')

  # Identify rows with values <= 0
  negAmount = pPf[pPf['cantidad_num'] <= 0]
  negPrices = pPf[pPf['precio_unitario_num'] <= 0]
  negTotales = pPf[pPf['total_num'] <= 0]
  negRows = pPf[(pPf['cantidad_num'] <= 0) | (pPf['precio_unitario_num'] <= 0) | (pPf['total_num'] <= 0)]

  # Show summary of results
  print(f"Rows with any negative or zero values: {len(negRows)}")
  print(f"Rows with [precio_unitario <= 0]:      {len(negPrices)}")
  print(f"Rows with [cantidad <= 0]:             {len(negAmount)}")
  print(f"Rows with [total <= 0]:                {len(negTotales)}")

  hlp.SectionBreak(0,0)
  print("Snapshot of rows with negative amounts or quantities:")
  print(negRows[['fecha', 'producto', 'cantidad', 'precio_unitario', 'total']].head(10))
  
hlp.SectionBegin("CHECK OF NEGATIVE OR ZERO NUMERIC VALUES", 0)
print("+ Summary of rows with weird numeric values:")
hlp.SectionBreak(0,0)
CheckNumericValues(df)
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)




+ =========================================================
+ CHECK OF NEGATIVE OR ZERO NUMERIC VALUES
+ Summary of rows with weird numeric values:
+ ---------------------------------------------------------
Rows with any negative or zero values: 30
Rows with [precio_unitario <= 0]:      10
Rows with [cantidad <= 0]:             11
Rows with [total <= 0]:                30
+ ---------------------------------------------------------
Snapshot of rows with negative amounts or quantities:
         fecha               producto  cantidad  precio_unitario   total
38  2026-10-08      Raton inalambrico      -2.0            36.76  -73.52
39  2026-09-29              Webcam HD      -1.0            78.83  -78.83
40  2026-10-11                Hub USB      -3.0            61.48 -184.44
41  2026-10-11    Monitor 24 pulgadas      -1.0           123.16 -123.16
42  2026-10-09           Funda tablet      -3.0            34.13 -102.39
43  2026-09-27       Soporte portatil      -1.0            45.54  -45.54

In [ ]:
# DATA CLEANING
# =============
# 6. CHECK OF OUTLIERS VALUES
# ---------------------------
tInit: float = time.time()

# CheckOutliersValues
# -------------------
# Convert columns ensuring numeric values and show the biggest 
# values to detect prices or quantities that makes no sense
# 
# @param pd.DataFrame pPf The pandas dataframe to check
# @return None
def CheckOutliersValues(pPf: pd.DataFrame) -> None:
  # Ensure numeric conversion
  pPf['qty_temp'] = pd.to_numeric(pPf['cantidad'], errors='coerce')
  pPf['prc_temp'] = pd.to_numeric(pPf['precio_unitario'], errors='coerce')
  
  # Show the highest values
  highestQty = pPf.nlargest(5, 'qty_temp')
  print(f"Quantity highest values: {len(highestQty)}")
  hlp.nbsp()
  print(highestQty[['fecha', 'producto', 'cantidad', 'precio_unitario', 'total']].head(5))
  hlp.SectionBreak(0,0)

  highestPrc = pPf.nlargest(5, 'prc_temp')
  print(f"Price highest values: {len(highestPrc)}")
  hlp.nbsp()
  print(highestPrc[['fecha', 'producto', 'cantidad', 'precio_unitario', 'total']].head(5))
  
hlp.SectionBegin("CHECK OF OUTLIERS VALUES", 0)
print("+ Summary of rows with possible outlier values on Price and Quantity:")
hlp.SectionBreak(0,0)
CheckOutliersValues(df)
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)


+ =========================================================
+ CHECK OF OUTLIERS VALUES
+ Summary of rows with possible outlier values on Price and Quantity:
+ ---------------------------------------------------------
Quantity highest values: 5
 
          fecha          producto  cantidad  precio_unitario     total
97   2026-10-15  Altavoz portatil     845.0           117.16  99000.20
94   2026-10-07  Teclado mecanico     794.0            44.07  34991.58
100  2026-09-25     Disco SSD 1TB     394.0           107.57  42382.58
96   2026-09-26      Funda tablet     315.0            18.08   5695.20
95   2026-09-29         Webcam HD     304.0            69.97  21270.88
+ ---------------------------------------------------------
Price highest values: 5
 
          fecha             producto  cantidad  precio_unitario     total
104  2026-09-29     Teclado mecanico       1.0          11922.0   11922.0
102  2026-10-08    Raton inalambrico       2.0          11838.0   23676.0
103  2026-09-19     